# 00 — Kaggle setup & environment smoke test

**AgriSense AI · Phase 0 (DEV_PLAN task 0.8, Kaggle edition)**

This notebook does two things:

1. Proves the Kaggle kernel is usable for this project — GPU present, the extra
   packages installed, data attached at `/kaggle/input`, secrets readable.
2. Writes a **setup report** artifact so an experiment run can be traced back to
   its environment.

Run it **once per notebook session** before any training notebook
(`02_disease_training.ipynb`, `04_yield_training.ipynb`).

> Before running: **Settings → Accelerator → GPU T4 x2 / P100**, and
> **Settings → Internet → On** (needed for `pip install` and artifact uploads).
> Kaggle does not persist files between sessions — see the persistence section
> at the end.

## 1. Environment report

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

REPORT = {}

REPORT["python"] = sys.version.split()[0]
REPORT["platform"] = platform.platform()
REPORT["cwd"] = os.getcwd()
REPORT["cpu_count"] = os.cpu_count()
REPORT["torch"] = torch.__version__
REPORT["cuda_available"] = torch.cuda.is_available()
REPORT["gpu"] = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        REPORT["gpu"].append(
            {
                "index": i,
                "name": props.name,
                "vram_gib": round(props.total_memory / 1024**3, 1),
                "capability": f"{props.major}.{props.minor}",
            }
        )

try:
    import psutil

    REPORT["ram_gib"] = round(psutil.virtual_memory().total / 1024**3, 1)
except ImportError:
    REPORT["ram_gib"] = None

for path in ("/kaggle/input", "/kaggle/working", "/kaggle/temp"):
    if Path(path).exists():
        usage = shutil.disk_usage(path)
        REPORT[f"disk_{path.replace('/', '_')}_free_gib"] = round(usage.free / 1024**3, 1)

print(json.dumps(REPORT, indent=2))

if not REPORT["cuda_available"]:
    print(
        "\n*** WARNING: no GPU. Set Settings -> Accelerator -> GPU before training. ***"
    )
    print("Refusing to continue would be safer, but the setup checks below still run on CPU.")

## 2. Install the packages Kaggle does not ship

In [ ]:
# module name -> pip requirement. Only missing packages are installed, so a
# preinstalled (and therefore GPU-matched) torch is never replaced.
REQUIRED = {
    "shap": "shap",
    "pytorch_grad_cam": "grad-cam",
    "optuna": "optuna",
    "timm": "timm",
    "albumentations": "albumentations",
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
    "yaml": "PyYAML",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "cv2": "opencv-python-headless",
}

import importlib.util

missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
print("missing packages:", missing or "none")

if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("installed.")

import importlib.metadata as package_metadata

REPORT["versions"] = {}
for dist in ("torch", "torchvision", "timm", "shap", "xgboost", "grad-cam"):
    try:
        REPORT["versions"][dist] = package_metadata.version(dist)
    except package_metadata.PackageNotFoundError:
        REPORT["versions"][dist] = None

print(json.dumps(REPORT["versions"], indent=2))

## 3. Configure this session

In [ ]:
CONFIG = {
    # Random seed for every library (recorded in the run file).
    "seed": 42,
    # Where attached Kaggle datasets appear. Glob so the slug can be anything.
   
    "dataset_glob": "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset",
    "data_root": "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color",

    # Everything that must survive the session goes here, then gets pushed out.
    "artifact_root": "/kaggle/working/artifacts",
    # Run name; the record is written under artifacts/runs/<run_id>/.
    "run_name": "setup-00-kaggle-smoke",
    # Optional: clone the repo when the code lives on GitHub instead of a dataset.
    "repo_url": "",
}

Path(CONFIG["artifact_root"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))


## 4. Locate the attached dataset

In [ ]:
import glob

candidates = sorted(glob.glob(CONFIG["dataset_glob"]))
data_root = Path(CONFIG["data_root"]) if CONFIG["data_root"] else (Path(candidates[0]) if candidates else None)
REPORT["data_root"] = str(data_root) if data_root else None

if data_root is None or not data_root.exists():
    print("No dataset found.")
    print("Attach it via the right-hand panel: Add Input -> Your Datasets.")
    print("Expected glob:", CONFIG["dataset_glob"])
else:
    print("data root:", data_root)
    entries = sorted(p.name for p in data_root.iterdir())[:20]
    print("top-level entries:", entries)

    images = list(data_root.rglob("*.jpg")) + list(data_root.rglob("*.JPG")) + list(data_root.rglob("*.png"))
    print(f"image files found: {len(images)}")
    if images:
        classes = sorted({p.parent.name for p in images})
        print(f"classes detected: {len(classes)}")
        for name in classes[:5]:
            print("  -", name)
        if len(classes) > 5:
            print("  ...")
    REPORT["image_count"] = len(images)

    # If the archive is nested one level (PlantVillage/PlantVillage/...), point deeper.
    if images and len({p.parent.name for p in images}) <= 1:
        for sub in sorted({p.parent.parent for p in images}):
            print("possible class directory:", sub)

## 5. Secrets (never printed, only checked for presence)

In [ ]:
def get_secret(name: str, default=None):
    """Read a Kaggle secret. Falls back to the env var for local runs."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, default)


SECRET_NAMES = [
    "SUPABASE_URL",
    "SUPABASE_S3_ACCESS_KEY_ID",
    "SUPABASE_S3_SECRET_ACCESS_KEY",
    "SUPABASE_S3_REGION",
    "SUPABASE_BUCKET_MODELS",
]

secrets = {name: get_secret(name) for name in SECRET_NAMES}
REPORT["secrets_present"] = {name: bool(value) for name, value in secrets.items()}
print(json.dumps(REPORT["secrets_present"], indent=2))

# Supabase exposes an S3-compatible endpoint under the project URL, so the
# endpoint never has to be stored as a separate secret.
STORAGE_ENDPOINT = (
    f"{secrets['SUPABASE_URL'].rstrip('/')}/storage/v1/s3"
    if secrets.get("SUPABASE_URL")
    else None
)
print("storage endpoint:", STORAGE_ENDPOINT or "<missing SUPABASE_URL>")

STORAGE_KEYS = ("SUPABASE_URL", "SUPABASE_S3_ACCESS_KEY_ID", "SUPABASE_S3_SECRET_ACCESS_KEY")
if not all(secrets.get(key) for key in STORAGE_KEYS):
    print("\nSupabase Storage secrets missing -> artifacts cannot be pushed out of this session.")
    print("Add them in Add-ons -> Secrets, then restart the session.")


## 6. Record the run (no tracking server)

In [ ]:
# No tracking server and no account: a run is a folder plus one line in an
# append-only index. The schema matches ml/src/common/tracking.py, so notebook
# runs and script runs end up in the same comparison table.
from datetime import UTC, datetime

started_at = datetime.now(UTC).isoformat(timespec="seconds")
run_id = f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}-{CONFIG['run_name']}"
run_dir = Path(CONFIG["artifact_root"]) / "runs" / run_id
run_dir.mkdir(parents=True, exist_ok=True)

record = {
    "run_id": run_id,
    "name": CONFIG["run_name"],
    "status": "completed",
    "started_at": started_at,
    "finished_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "params": {"seed": CONFIG["seed"], "data_root": REPORT.get("data_root")},
    "metrics": {"image_count": float(REPORT.get("image_count") or 0)},
    "tags": {
        "host": "kaggle",
        "gpu": REPORT["gpu"][0]["name"] if REPORT["gpu"] else "cpu",
    },
    "artifacts": ["run.json"],
    "environment": REPORT,
}
(run_dir / "run.json").write_text(json.dumps(record, indent=2), encoding="utf-8")

summary = {
    key: record[key]
    for key in ("run_id", "name", "status", "started_at", "finished_at", "metrics")
}
summary["run_dir"] = str(run_dir)
with (run_dir.parent / "index.jsonl").open("a", encoding="utf-8") as handle:
    handle.write(json.dumps(summary) + "\n")

REPORT["run_id"] = run_id
print("run dir:", run_dir)
print("record written:", (run_dir / "run.json").is_file())


## 7. Determinism

In [ ]:
import random

import numpy as np


def seed_everything(seed: int = 42, deterministic: bool = False) -> int:
    """Seed every RNG. `deterministic=True` also forces deterministic kernels.

    Mirrors ml/src/common/seed.py — that module is the source of truth for
    training scripts; this copy keeps the notebook standalone.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.use_deterministic_algorithms(True, warn_only=True)
        torch.backends.cudnn.benchmark = False
    return seed


print("seed:", seed_everything(CONFIG["seed"]))

## 8. GPU smoke test (forward + backward + timing)

In [ ]:
import time

import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torchvision.models.resnet18(weights=None).to(device)
model.train()

batch_size, image_size = 32, 224
x = torch.randn(batch_size, 3, image_size, image_size, device=device)
y = torch.randint(0, 10, (batch_size,), device=device)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for step in range(3):  # warm-up
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()

if device.type == "cuda":
    torch.cuda.synchronize()

start = time.perf_counter()
steps = 10
for _ in range(steps):
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()

if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start

images_per_s = steps * batch_size / elapsed
print(f"device: {device}")
print(f"throughput: {images_per_s:.1f} images/s ({elapsed / steps * 1000:.1f} ms/step)")
print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB" if device.type == "cuda" else "")

REPORT["smoke_test"] = {
    "device": str(device),
    "images_per_s": round(images_per_s, 1),
    "batch_size": batch_size,
}
assert torch.isfinite(loss), "loss is not finite — environment is broken"
print("OK — training loop works on this kernel.")

## 9. Persist artifacts (optional, needs Supabase Storage secrets)

In [ ]:
def push_dir_to_storage(local_dir: str, prefix: str) -> None:
    """Upload a directory to Supabase Storage through its S3-compatible API.

    Used to survive the session: Kaggle keeps no disk between runs.
    """
    import boto3

    required = ("SUPABASE_URL", "SUPABASE_S3_ACCESS_KEY_ID", "SUPABASE_S3_SECRET_ACCESS_KEY")
    if not all(secrets.get(key) for key in required):
        print("Supabase Storage secrets missing - skipping upload.")
        return

    client = boto3.client(
        "s3",
        endpoint_url=STORAGE_ENDPOINT,
        aws_access_key_id=secrets["SUPABASE_S3_ACCESS_KEY_ID"],
        aws_secret_access_key=secrets["SUPABASE_S3_SECRET_ACCESS_KEY"],
        # Must match the project region or Supabase rejects the signature.
        region_name=secrets.get("SUPABASE_S3_REGION") or "us-east-1",
    )
    bucket = secrets.get("SUPABASE_BUCKET_MODELS") or "agri-models"
    root = Path(local_dir)
    uploaded = 0
    for path in root.rglob("*"):
        if path.is_file():
            key = f"{prefix}/{path.relative_to(root).as_posix()}"
            client.upload_file(str(path), bucket, key)
            uploaded += 1
    print(f"uploaded {uploaded} files to {bucket}/{prefix}/")


# Uncomment once artifacts exist:
# push_dir_to_storage(CONFIG["artifact_root"], "runs/setup-00")
print("helper ready.")


## 10. What survives this session

| Thing | Survives? | Notes |
|---|---|---|
| Files in `/kaggle/working` | Only if you **Save Version → Save & Run All (Commit)** | 20 GB output limit |
| `/kaggle/temp` | No | Scratch space only |
| Installed pip packages | No | Reinstall each session (cell 2 is idempotent) |
| Run records under `artifacts/runs/` | Only via the commit above | Push `runs/` to Supabase Storage with the artifacts |
| Training checkpoints | No | Push to Supabase Storage or a Kaggle Dataset version every epoch |
| Kaggle secrets | Yes | Attached per notebook, not per session |

**Session limits to plan around** (verify in the Kaggle UI — quotas change):
~12 h max runtime, ~30 GPU-hours/week, 20 GB `/kaggle/working`, and a kernel is
killed outright at the limit with no warning. Training scripts must checkpoint
every epoch and be resumable (DEV_PLAN §2.5, risk R5).

**A killed session is the normal failure mode.** Anything not pushed out or
committed is gone.

## 11. Next steps

1. Attach the PlantVillage dataset → re-run cells 4–5.
2. `ml/notebooks/kaggle/01_plantvillage_eda.ipynb` (Phase 1) — class counts,
   duplicates, splits, `data_manifest.json`.
3. `ml/configs/disease/*.yaml` (Phase 2) — one config per experiment D-01…D-06.
4. Keep this notebook's `setup_report.json` with every reported run.